# T06 · EDA: colesterol, glicose e hábitos de vida

**Issue:** #6 · **Fase 1 · EDA** · **Dataset:** `data/raw/cardio_train.csv`

**Objetivo:** analisar os exames laboratoriais (`cholesterol` e `gluc`, que são **ordinais**: 1 = normal, 2 = acima do normal,
3 = muito acima) e os hábitos **autodeclarados** pelo paciente (`smoke`, `alco` e `active`). A análise alimenta o
tratamento ordinal dessas colunas no pré-processamento (**T11**) e a discussão crítica do relatório (**T34**).

> A análise usa os dados **limpos** (`load_dataset()` + `clean()`), o mesmo conjunto que os modelos veem.
> Todos os números dos textos são **calculados a partir do dataset**.
> É uma análise descritiva: mostra **associação**, não causa.

In [1]:
import sys
from pathlib import Path

# Permite rodar o notebook fora do Docker (no Docker o PYTHONPATH já aponta para src/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from cardio.config import DATA_PATH, FIGURES_DIR
from cardio.data.loader import load_dataset
from cardio.data.preprocessing import clean

sns.set_theme(style="whitegrid")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
NIVEIS = {1: "Normal", 2: "Acima do normal", 3: "Muito acima"}
SEXO = {0: "Mulheres", 1: "Homens"}
HABITOS = {"smoke": "Fuma", "alco": "Bebe álcool", "active": "Atividade física"}


def salvar(fig, nome: str) -> None:
    """Salva a figura em results/figures/ com o prefixo da tarefa."""
    fig.savefig(FIGURES_DIR / f"t06_{nome}.png", dpi=120, bbox_inches="tight")


def pct(valor: float, casas: int = 0) -> str:
    """Formata uma proporção (0–1) como porcentagem no padrão brasileiro: 0.497 -> '49,7%'."""
    return f"{valor * 100:.{casas}f}%".replace(".", ",")


def num(valor: float, casas: int = 2) -> str:
    """Formata um número com vírgula decimal: 0.238 -> '0,24'."""
    return f"{valor:.{casas}f}".replace(".", ",")


def inteiro(valor: int) -> str:
    """Formata um inteiro com ponto de milhar: 68588 -> '68.588'."""
    return f"{valor:,}".replace(",", ".")


def texto(conteudo: str) -> None:
    """Mostra um texto de interpretação formatado em Markdown."""
    display(Markdown(conteudo))

In [2]:
df = clean(load_dataset(DATA_PATH), verbose=False)
df.attrs = {}  # o log da limpeza (um DataFrame) não deve acompanhar as tabelas derivadas
df["sexo"] = pd.Categorical(df["sex_male"].map(SEXO), categories=list(SEXO.values()))
TAXA_GERAL = df["cardio"].mean()

print(f"{inteiro(len(df))} pacientes (após a limpeza) | {pct(TAXA_GERAL, 1)} com doença cardiovascular")
df[["cholesterol", "gluc", "smoke", "alco", "active"]].describe().round(2)

68.509 pacientes (após a limpeza) | 49,5% com doença cardiovascular


,cholesterol,gluc,smoke,alco,active
count,68509.00,68509.00,68509.00,68509.00,68509.0
mean,1.36,1.23,0.09,0.05,0.8
std,0.68,0.57,0.28,0.22,0.4
min,1.00,1.00,0.00,0.00,0.0
25%,1.00,1.00,0.00,0.00,1.0
50%,1.00,1.00,0.00,0.00,1.0
75%,2.00,1.00,0.00,0.00,1.0
max,3.00,3.00,1.00,1.00,1.0


## 1. Colesterol e glicose × doença

`cholesterol` e `gluc` vêm do exame do paciente, em três categorias ordenadas. O gráfico mostra a **% de pacientes com doença
em cada nível** e quantos pacientes há em cada um.

In [3]:
def taxa_por_nivel(coluna: str) -> pd.DataFrame:
    """Taxa de doença e número de pacientes em cada nível (1, 2 e 3) da coluna."""
    tabela = df.groupby(coluna)["cardio"].agg(taxa="mean", n="size")
    tabela.index = tabela.index.map(NIVEIS)
    return tabela


taxa_col, taxa_glic = taxa_por_nivel("cholesterol"), taxa_por_nivel("gluc")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, tabela, titulo in ((axes[0], taxa_col, "colesterol"), (axes[1], taxa_glic, "glicose")):
    (tabela["taxa"] * 100).plot(kind="bar", ax=ax, color="tab:red", legend=False)
    ax.axhline(TAXA_GERAL * 100, color="gray", linestyle="--", label="Taxa geral")
    ax.set(title=f"Taxa de doença por nível de {titulo}", xlabel="", ylabel="% com doença", ylim=(0, 100))
    ax.set_xticks(range(len(tabela)), [f"{nivel}\n(n = {inteiro(n)})" for nivel, n in zip(tabela.index, tabela["n"])], rotation=0)
    for i, taxa in enumerate(tabela["taxa"]):
        ax.text(i, taxa * 100 / 2, pct(taxa, 1), ha="center", va="center", color="white", fontweight="bold")
axes[0].legend(handles=[axes[0].lines[0]], loc="upper left")

salvar(fig, "colesterol_glicose")
plt.show()

In [4]:
rho_col = df["cholesterol"].corr(df["cardio"], method="spearman")
rho_glic = df["gluc"].corr(df["cardio"], method="spearman")
rho_entre = df["cholesterol"].corr(df["gluc"], method="spearman")
sobe_col = taxa_col["taxa"].is_monotonic_increasing
sobe_glic = taxa_glic["taxa"].is_monotonic_increasing
salto_col = (taxa_col["taxa"].iloc[-1] - taxa_col["taxa"].iloc[0]) * 100
salto_glic = (taxa_glic["taxa"].iloc[-1] - taxa_glic["taxa"].iloc[0]) * 100
total = len(df)

texto(f"""**Interpretação:** {"nas duas variáveis, a taxa de doença **cresce a cada nível**" if sobe_col and sobe_glic
else "a taxa de doença não cresce em ordem nas duas variáveis"}.

- **Colesterol:** a taxa vai de **{pct(taxa_col["taxa"].iloc[0], 1)}** (normal) a **{pct(taxa_col["taxa"].iloc[-1], 1)}** (muito acima),
  uma diferença de {num(salto_col, 1)} pontos percentuais (correlação de Spearman com o alvo: {num(rho_col)}).
- **Glicose:** vai de **{pct(taxa_glic["taxa"].iloc[0], 1)}** a **{pct(taxa_glic["taxa"].iloc[-1], 1)}**,
  uma diferença de {num(salto_glic, 1)} pontos percentuais (correlação de Spearman: {num(rho_glic)}).

{"O colesterol se associa mais à doença do que a glicose." if salto_col > salto_glic else "A glicose se associa mais à doença do que o colesterol."}
Colesterol e glicose altos tendem a aparecer juntos (correlação de Spearman entre eles: {num(rho_entre)}), então parte do
efeito de uma pode estar misturada com o da outra.

Como o risco **aumenta em ordem** a cada nível, faz sentido tratar as duas colunas como **ordinais** (1 < 2 < 3),
mantendo os valores como estão, em vez de transformá-las em categorias independentes (one-hot), como é feito na T11.
Os níveis "acima do normal" e "muito acima" são grupos pequenos (colesterol: {pct(taxa_col["n"].iloc[1] / total)} e
{pct(taxa_col["n"].iloc[2] / total)} dos pacientes; glicose: {pct(taxa_glic["n"].iloc[1] / total)} e
{pct(taxa_glic["n"].iloc[2] / total)}), então suas taxas são menos precisas que a do nível normal.""")

**Interpretação:** nas duas variáveis, a taxa de doença **cresce a cada nível**.

- **Colesterol:** a taxa vai de **43,6%** (normal) a **76,2%** (muito acima),
  uma diferença de 32,7 pontos percentuais (correlação de Spearman com o alvo: 0,22).
- **Glicose:** vai de **47,6%** a **61,7%**,
  uma diferença de 14,1 pontos percentuais (correlação de Spearman: 0,09).

O colesterol se associa mais à doença do que a glicose.
Colesterol e glicose altos tendem a aparecer juntos (correlação de Spearman entre eles: 0,41), então parte do
efeito de uma pode estar misturada com o da outra.

Como o risco **aumenta em ordem** a cada nível, faz sentido tratar as duas colunas como **ordinais** (1 < 2 < 3),
mantendo os valores como estão, em vez de transformá-las em categorias independentes (one-hot), como é feito na T11.
Os níveis "acima do normal" e "muito acima" são grupos pequenos (colesterol: 14% e
11% dos pacientes; glicose: 7% e
8%), então suas taxas são menos precisas que a do nível normal.

## 2. Hábitos de vida × doença

`smoke`, `alco` e `active` são **autodeclarados** (0 = não, 1 = sim). O gráfico mostra a taxa de doença em quem declara
"não" e em quem declara "sim" para cada hábito.

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), sharey=True)
taxas = {}
for ax, (coluna, nome) in zip(axes, HABITOS.items()):
    tabela = df.groupby(coluna)["cardio"].agg(taxa="mean", n="size")
    tabela.index = tabela.index.map({0: "Não", 1: "Sim"})
    taxas[coluna] = tabela
    (tabela["taxa"] * 100).plot(kind="bar", ax=ax, color=["tab:blue", "tab:orange"], legend=False)
    ax.axhline(TAXA_GERAL * 100, color="gray", linestyle="--", label="Taxa geral")
    ax.set(title=f"{nome} ({pct(df[coluna].mean(), 1)} dizem sim)", xlabel="", ylabel="% com doença", ylim=(0, 100))
    ax.set_xticks(range(len(tabela)), [f"{resposta}\n(n = {inteiro(n)})" for resposta, n in zip(tabela.index, tabela["n"])], rotation=0)
    for i, taxa in enumerate(tabela["taxa"]):
        ax.text(i, taxa * 100 / 2, pct(taxa, 1), ha="center", va="center", color="white", fontweight="bold")
axes[0].legend(handles=[axes[0].lines[0]], loc="upper left")

salvar(fig, "habitos")
plt.show()

In [6]:
linhas = []
for coluna, nome in HABITOS.items():
    tabela = taxas[coluna]
    diferenca = (tabela.loc["Sim", "taxa"] - tabela.loc["Não", "taxa"]) * 100
    linhas.append(
        f"- **{nome}** ({pct(df[coluna].mean(), 1)} dos pacientes): taxa de doença de {pct(tabela.loc['Sim', 'taxa'], 1)} em quem declara "
        f"'sim' contra {pct(tabela.loc['Não', 'taxa'], 1)} em quem declara 'não', ou seja, **{num(abs(diferenca), 1)} pontos "
        f"{'maior' if diferenca > 0 else 'menor'}** (correlação com o alvo: {num(df[coluna].corr(df['cardio']), 3)})."
    )

contra = [{"smoke": "fumo", "alco": "álcool"}[c] for c in ("smoke", "alco") if taxas[c].loc["Sim", "taxa"] < taxas[c].loc["Não", "taxa"]]
protege = taxas["active"].loc["Sim", "taxa"] < taxas["active"].loc["Não", "taxa"]
texto("**Interpretação:**\n\n" + "\n".join(linhas) + f"""

{"A **atividade física** tem o sinal esperado: quem a pratica tem menos doença." if protege else "A atividade física não tem o sinal esperado."}
{f"**{' e '.join(contra).capitalize()}** saem com sinal **contrário ao esperado**: quem declara o hábito tem **menos** doença, e a correlação é muito fraca (próxima de zero)." if contra else "Fumo e álcool têm o sinal esperado."}
Isso não quer dizer que fumar ou beber proteja o coração: a seção seguinte investiga o resultado.""")

**Interpretação:**

- **Fuma** (8,8% dos pacientes): taxa de doença de 46,9% em quem declara 'sim' contra 49,7% em quem declara 'não', ou seja, **2,9 pontos menor** (correlação com o alvo: -0,016).
- **Bebe álcool** (5,3% dos pacientes): taxa de doença de 47,7% em quem declara 'sim' contra 49,6% em quem declara 'não', ou seja, **1,9 pontos menor** (correlação com o alvo: -0,009).
- **Atividade física** (80,3% dos pacientes): taxa de doença de 48,6% em quem declara 'sim' contra 53,3% em quem declara 'não', ou seja, **4,7 pontos menor** (correlação com o alvo: -0,038).

A **atividade física** tem o sinal esperado: quem a pratica tem menos doença.
**Fumo e álcool** saem com sinal **contrário ao esperado**: quem declara o hábito tem **menos** doença, e a correlação é muito fraca (próxima de zero).
Isso não quer dizer que fumar ou beber proteja o coração: a seção seguinte investiga o resultado.

## 3. O resultado contraintuitivo do fumo

Fumar é um fator de risco cardiovascular bem estabelecido, então uma taxa **menor** entre os fumantes pede investigação.
A primeira hipótese é **confusão com sexo e idade**: se os fumantes forem, por exemplo, mais jovens ou de um sexo com menos
doença, a comparação geral engana. Para testar, comparamos fumantes e não fumantes **dentro de cada sexo e de cada faixa etária**.

In [7]:
df["faixa_etaria"] = pd.cut(df["age_years"], [0, 45, 50, 55, 60, 70], right=False,
                            labels=["< 45", "45–49", "50–54", "55–59", "60+"])
NOMES_FUMO = {0: "Não fuma", 1: "Fuma"}

por_sexo = df.pivot_table(index="sexo", columns="smoke", values="cardio", observed=True).rename(columns=NOMES_FUMO)
por_idade = df.pivot_table(index="faixa_etaria", columns="smoke", values="cardio", observed=True).rename(columns=NOMES_FUMO)
n_idade = df.pivot_table(index="faixa_etaria", columns="smoke", values="cardio", aggfunc="size", observed=True).rename(columns=NOMES_FUMO)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, tabela, titulo in ((axes[0], por_sexo, "sexo"), (axes[1], por_idade, "faixa etária")):
    (tabela * 100).plot(kind="bar", ax=ax, color=["tab:blue", "tab:orange"])
    ax.axhline(TAXA_GERAL * 100, color="gray", linestyle="--")
    ax.set(title=f"Taxa de doença de fumantes e não fumantes, por {titulo}", xlabel="", ylabel="% com doença", ylim=(0, 100))
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="")
for container in axes[0].containers:
    axes[0].bar_label(container, labels=[pct(v / 100, 1) for v in container.datavalues],
                      label_type="center", color="white", fontweight="bold")

salvar(fig, "fumo_estratificado")
plt.show()

print("Taxa de doença (%) e número de pacientes em cada grupo (faixa etária × fuma):")
pd.concat({"% com doença": por_idade * 100, "pacientes": n_idade}, axis=1).round(1)

Taxa de doença (%) e número de pacientes em cada grupo (faixa etária × fuma):


% com doença       pacientes      
smoke            Não fuma  Fuma  Não fuma  Fuma
faixa_etaria                                   
< 45                 29.0  31.1      8962  1113
45–49                42.6  42.9      9999  1171
50–54                46.2  45.0     15317  1410
55–59                56.5  53.7     16606  1447
60+                  67.0  63.7     11592   892

In [8]:
prevalencia_sexo = df.groupby("sexo", observed=True)["smoke"].mean()
idade_media = df.groupby("smoke")["age_years"].mean()
menor_sexo = int((por_sexo["Fuma"] < por_sexo["Não fuma"]).sum())
menor_idade = int((por_idade["Fuma"] < por_idade["Não fuma"]).sum())
dif_idade = (por_idade["Fuma"] - por_idade["Não fuma"]) * 100
geral = taxas["smoke"]["taxa"]

texto(f"""**Interpretação:**

- **Geral:** fumantes têm {pct(geral["Sim"], 1)} de doença contra {pct(geral["Não"], 1)} dos não fumantes.
- **Sexo:** fumar é muito mais comum entre os homens ({pct(prevalencia_sexo["Homens"], 1)}) do que entre as mulheres
  ({pct(prevalencia_sexo["Mulheres"], 1)}), mas **dentro de cada sexo** o fumante tem menos doença em {menor_sexo} dos {len(por_sexo)} grupos
  (homens: {pct(por_sexo.loc["Homens", "Fuma"], 1)} × {pct(por_sexo.loc["Homens", "Não fuma"], 1)}; mulheres:
  {pct(por_sexo.loc["Mulheres", "Fuma"], 1)} × {pct(por_sexo.loc["Mulheres", "Não fuma"], 1)}). O sexo, sozinho, não explica o resultado.
- **Idade:** os fumantes são um pouco {"mais jovens" if idade_media[1] < idade_media[0] else "mais velhos"} ({num(idade_media[1], 1)} contra
  {num(idade_media[0], 1)} anos em média) e o fumante tem menos doença em {menor_idade} das {len(por_idade)} faixas etárias.
  A diferença de taxa (fuma − não fuma, em pontos percentuais; negativo = fumante com menos doença) é de {"; ".join(f"{faixa}: {valor:+.1f}".replace(".", ",") for faixa, valor in dif_idade.items())}.
  Nas faixas mais velhas o fumante continua com menos doença, então a idade explica, no máximo, uma parte pequena do resultado.""")

**Interpretação:**

- **Geral:** fumantes têm 46,9% de doença contra 49,7% dos não fumantes.
- **Sexo:** fumar é muito mais comum entre os homens (21,9%) do que entre as mulheres
  (1,8%), mas **dentro de cada sexo** o fumante tem menos doença em 2 dos 2 grupos
  (homens: 46,9% × 50,8%; mulheres:
  46,9% × 49,3%). O sexo, sozinho, não explica o resultado.
- **Idade:** os fumantes são um pouco mais jovens (52,3 contra
  53,4 anos em média) e o fumante tem menos doença em 3 das 5 faixas etárias.
  A diferença de taxa (fuma − não fuma, em pontos percentuais; negativo = fumante com menos doença) é de < 45: +2,0; 45–49: +0,3; 50–54: -1,2; 55–59: -2,8; 60+: -3,3.
  Nas faixas mais velhas o fumante continua com menos doença, então a idade explica, no máximo, uma parte pequena do resultado.

**Hipóteses que sobram** (os dados **não permitem testá-las**, porque não há histórico de tabagismo nem a data do diagnóstico):

- **Autodeclaração:** quem fuma pode omitir o hábito (subnotificação), e a omissão pode ser diferente entre quem tem e quem
  não tem doença.
- **Causalidade reversa:** pacientes que pararam de fumar **depois do diagnóstico** passam a se declarar não fumantes. Isso
  empurra os casos de doença para o grupo "não fuma" e deixa o grupo "fuma" com menos doença.
- **Medida pobre do hábito:** `smoke` é sim/não. Não mede quantidade nem há quanto tempo a pessoa fuma, que é o que importa
  clinicamente.
- **Confusão com sexo e idade:** testada acima. Explica só uma parte do resultado.

> **Conclusão:** o dado **não** indica que fumar proteja o coração. O sinal fraco e contrário ao esperado vem de como a
> variável foi coletada. Ela pode ficar no modelo pelo valor preditivo que tiver, mas o seu efeito não deve ser lido como
> relação causal nas interpretações (T24, T25) nem na discussão crítica (T34).

## 4. Hábitos por sexo

Fumo e álcool são muito mais comuns entre os homens? Isso também importa para a análise por subgrupo (T26).

In [9]:
habitos_sexo = df.groupby("sexo", observed=True)[list(HABITOS)].mean().rename(columns=HABITOS)

fig, ax = plt.subplots(figsize=(8.5, 4.5))
(habitos_sexo.T * 100).plot(kind="bar", ax=ax, color=["tab:purple", "tab:blue"])
ax.set(title="Hábitos declarados, por sexo", xlabel="", ylabel="% que declara 'sim'", ylim=(0, 100))
ax.tick_params(axis="x", rotation=0)
ax.legend(title="")
for container in ax.containers:
    ax.bar_label(container, labels=[pct(v / 100, 1) for v in container.datavalues], padding=2)

salvar(fig, "habitos_por_sexo")
plt.show()

In [10]:
razao_fumo = habitos_sexo.loc["Homens", "Fuma"] / habitos_sexo.loc["Mulheres", "Fuma"]
razao_alcool = habitos_sexo.loc["Homens", "Bebe álcool"] / habitos_sexo.loc["Mulheres", "Bebe álcool"]
share_fumantes_homens = df.loc[df["smoke"] == 1, "sex_male"].mean()

texto(f"""**Interpretação:** sim. Os homens fumam **{num(razao_fumo, 1)} vezes mais** que as mulheres
({pct(habitos_sexo.loc["Homens", "Fuma"], 1)} × {pct(habitos_sexo.loc["Mulheres", "Fuma"], 1)}) e bebem **{num(razao_alcool, 1)} vezes mais**
({pct(habitos_sexo.loc["Homens", "Bebe álcool"], 1)} × {pct(habitos_sexo.loc["Mulheres", "Bebe álcool"], 1)}).
{pct(share_fumantes_homens)} dos fumantes são homens. A atividade física é parecida nos dois sexos
({pct(habitos_sexo.loc["Mulheres", "Atividade física"], 1)} × {pct(habitos_sexo.loc["Homens", "Atividade física"], 1)}).

Como a amostra tem **{pct(1 - df["sex_male"].mean())} de mulheres**, as conclusões sobre fumo vêm em grande parte dos homens
e as de álcool também pesam mais neles; é uma limitação a registrar na discussão crítica (T34) e na análise por subgrupo (T26).""")

**Interpretação:** sim. Os homens fumam **12,3 vezes mais** que as mulheres
(21,9% × 1,8%) e bebem **4,2 vezes mais**
(10,6% × 2,5%).
87% dos fumantes são homens. A atividade física é parecida nos dois sexos
(80,2% × 80,7%).

Como a amostra tem **65% de mulheres**, as conclusões sobre fumo vêm em grande parte dos homens
e as de álcool também pesam mais neles; é uma limitação a registrar na discussão crítica (T34) e na análise por subgrupo (T26).

## 5. Confiabilidade das variáveis e conclusões

In [11]:
texto(f"""### Confiabilidade

- **Colesterol e glicose** vêm do **exame** do paciente, então são mais confiáveis, mas chegam só em 3 categorias
  (perdem o valor medido) e os níveis altos são grupos pequenos.
- **Fumo, álcool e atividade física** são **autodeclarados**: dependem do que o paciente conta. Costumam ser
  **subnotificados** (quem fuma ou bebe tende a omitir) e podem mudar com o diagnóstico (quem parou de fumar depois de
  adoecer se declara não fumante). Por isso o sinal contrário ao esperado de fumo e álcool deve ser lido como um **limite do
  dado**, não como um efeito protetor.

### Conclusões

- **Colesterol** é a variável mais associada à doença entre as analisadas aqui: a taxa vai de {pct(taxa_col["taxa"].iloc[0])} a
  {pct(taxa_col["taxa"].iloc[-1])}. A **glicose** vai de {pct(taxa_glic["taxa"].iloc[0])} a {pct(taxa_glic["taxa"].iloc[-1])}.
  O risco cresce em ordem nas duas, o que **sustenta tratá-las como ordinais** na T11.
- **Atividade física** tem o sinal esperado; **fumo** e **álcool** têm associação muito fraca e de sinal contrário ao esperado,
  que **não** se explica só por sexo e idade (T34).
- Os hábitos são sim/não e autodeclarados: a T11 os mantém como binários, e as interpretações do modelo (T24 e T25) precisam
  de cautela com `smoke` e `alco`.
- A amostra tem {pct(1 - df["sex_male"].mean())} de mulheres e fumo e álcool são concentrados nos homens: limitação para a T26 e a T34.""")

### Confiabilidade

- **Colesterol e glicose** vêm do **exame** do paciente, então são mais confiáveis, mas chegam só em 3 categorias
  (perdem o valor medido) e os níveis altos são grupos pequenos.
- **Fumo, álcool e atividade física** são **autodeclarados**: dependem do que o paciente conta. Costumam ser
  **subnotificados** (quem fuma ou bebe tende a omitir) e podem mudar com o diagnóstico (quem parou de fumar depois de
  adoecer se declara não fumante). Por isso o sinal contrário ao esperado de fumo e álcool deve ser lido como um **limite do
  dado**, não como um efeito protetor.

### Conclusões

- **Colesterol** é a variável mais associada à doença entre as analisadas aqui: a taxa vai de 44% a
  76%. A **glicose** vai de 48% a 62%.
  O risco cresce em ordem nas duas, o que **sustenta tratá-las como ordinais** na T11.
- **Atividade física** tem o sinal esperado; **fumo** e **álcool** têm associação muito fraca e de sinal contrário ao esperado,
  que **não** se explica só por sexo e idade (T34).
- Os hábitos são sim/não e autodeclarados: a T11 os mantém como binários, e as interpretações do modelo (T24 e T25) precisam
  de cautela com `smoke` e `alco`.
- A amostra tem 65% de mulheres e fumo e álcool são concentrados nos homens: limitação para a T26 e a T34.